- Nombre: Santiago Silveira

---
## **Implementación de un Asistente Inteligente para responder preguntas sobre documentos**

### Análisis del problema

Se requiere construir un asistente que responda preguntas basadas en un conjunto de documentos, específicamente el dataset *HotpotQA*. La característica distintiva de este dataset es que requiere razonamiento **multi-hop**, lo que implica que para responder correctamente, el sistema debe conectar información de múltiples documentos.

Debido al dinamismo de los documentos, no será suficiente utilizar un LLM como único componente del sistema. Se propone entonces utilizar **Retrieval-Augmented Generation (RAG)** para mejorar la calidad y precisión de las respuestas.

---
### Diseño del sistema

**Motivación**:
Los grandes modelos de lenguaje son entrenados con enromes cantidades de datos, pero en ocasiones, sus respuestas pueden ser equivocadas o estar obsoletas debido al dinamismo de la propia información. RAG resuelve este problema agregando nuestros propios datos a la que el LLM ya tiene acceso.

En RAG nuestros datos son cargados y preparados para ser "indexados". Las consultas de los usuarios son utilizadas sobre ese índice para buscar el contexto más relevante. Este contexto y la consulta son enviados al LLM, el cual nos va a proveer de una respuesta basada en datos en los cuales nosotros podemos confiar.

Para este sistema se propne la implementación de un *RAG pipeline* que lleve a cabo las fases necesarias para cumplir con los requerimientos, utilzando el framework *LlamaIndex*.

LlamaIndex es un framework para crear aplicaciones potenciadas por LLMs. Permite crear agentes, workflows, pipelines RAG, etc.

Algunos casos de uso de este framework son:
- Question-Answering (Retrieval-Augmented Generation aka RAG)
- Chatbots
- Document Understanding and Data Extraction
- Autonomous Agents that can perform research and take actions

entre otros.

En el siguiente diagrama se ilustran las cinco etapas clave en la técnica RAG. 

![Diagrama Pipeline RAG](.\images\rag-pipeline.png "Pipeline RAG")

Una idea sobre de cada fase:

- Carga: Leer nuestros documentos y transformarlos de forma tal que el sistema los pueda procesar.
- Indexación: Generar representaciones numéricas de los documentos para permitir una búsqueda eficiente.
- Almacenamiento: La representación numérica de los datos se guarda en una base de datos vectorial. Nos permite recuperar la información sin tener que recalcular la representación en un futuro.
- Consultas: Aceptar consultas del usuario y devolver las respuestas.
- Evaluación: Medir la calidad de las respuestas generadas (¿qué tan bueno es nuestro sistema?)

**Componentes del sistema**:

- Query Engine: 
- Retriever: 
- Embeddings model: 
- Response Synthesizer: 
- Index: 
- Vector Databse: 
- Collection: 
- LLM: 

El siguiente diagrama resume la conexión de estos componentes, así como también la interacción entre ellos en el flujo de datos que se dispara a través de una consulta del usuario:

![Sistema](.\images\system.png "Diagrama del Sistema")

El **retriever**, aunque a veces implícito, siempre está presente porque es el componente fundamental que decide qué partes de los documentos son relevantes para cada pregunta antes de que estas sean enviadas al LLM. Lo mismo ocurre con el **response synthesizer**.

**¿Cómo funcionan en conjunto?**

1. El usuario hace una consulta a través del Query Engine.
2. El Query Engine recibe la consulta y la pasa al Retriever.
3. El Retriever:
    - Convierte la consulta a un vector de embeddings
    - Busca en el índice los documentos más similares usando, por ejemplo, smilitud coseno
    - Devuelve los nodos (objetos `Node`) más relevantes al Query Engine
4. El Query Engine pasa:
    - La consulta original
    - Los nodos recuperados al Response Synthesizer
5. El Response Sythesizer:
    - Toma los nodos y extrae su contenido textual
    - Construye un prompt estructurado que incluye:
        - El contexto (o sea, contenido de los nodos)
        - La pregunta del usuario
        - Instrucciones para el LLM
    - Aplica su estrategia de síntesis (refine/compact/tree/simple)
6. El Response Synthesizer envía el prompt al LLM
7. El LLM:
    - Procesa el prompt
    - Genera una respuesta basada en el contexto proporcionado
    - Devuelve la respuesta al Response Synthesizer
8. El Response Synthesizer:
    - La devuelve al Query Engine
9. El Query Engine devuelve la respuesta final al usuario

A continuación sigue el proceso de implementación, acompañado de explicaciones sobre el proceso y las decisiones tomadas.

---
### Importando las Librerías

In [1]:
# De uso general
import json
import os
from IPython.display import clear_output
from typing import List
from dotenv import load_dotenv

# Cargar variables de entorno
load_dotenv()

# Componentes de LlamaIndex
from llama_index.core import (
    Document,
    VectorStoreIndex,
    Settings,
    StorageContext)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.llms.openai import OpenAI

import chromadb

clear_output()

---
### Carga e ingestión de los datos

Lo primero que debemos hacer es procesar nuestros datos y cargarlos. Para facilitar la tarea, contamos con un archivo JSON que tiene todos los documentos que se deben indexar: `hotpotqa_docs_reduced.json`.

Si exploramos un poco este archivo, podremos notar que refleja la naturaleza multihop de HotpotQA. En este dataset, una única pregunta se asocia a mútiples documentos. Cada entrada o elemento del archvo incluye:
- Question/Answer: representan la pregunta global y su respuesta correcta.
- Title/Text: El título y el texto corresponden a un artículo. Este fragmento puede ser una fuente de informació relevante o un distractor. O sea, no siempre contiene de manera directa la respuesta, sino que aporta parte de la evidencia necesaria para realizar el razonamiento multihop.

La siguiente función lee los elementos del json y por cada uno crea un documento de LlamaIndex (clase `Document`).

Para enriquecer el contexto del documento, no solo se va a incluir el campo `text` en cada documento, sino también `question`, `answer` y `title`. Incluyendo todos los campos pretendo proporcionarle al sistema más información y metadatos que pueden ser útiles para:
1. **Mejorar la relevancia en la búsqueda**, puesto que el motor de recuperación tiene más datos para comparar y relacionar con la consulta del usuario. Por ejemplo, el título o incluso la pregunta orignial del dataset pueden contener pistas importantes sobre el contenido del documento.
2. **Contextualizar la información**, dado que en escenarios de *multi-hop reasoning*, la relación entre la pregunta original, la respuesta y el contenido textual clave. La combianción de estos campos ayuda a que el modelo entienda el contexto completo y genere respuestas más precisas.

En este caso, como los documentos son relativamente cortos y están en texto plano, no se va a realizar ningún parseo especial de los nodos, y por cada item del json, se creará un documento.

In [ ]:
def load_documents_from_json(file_path: str) -> List[Document]:
    '''
    Loads documents from a JSON file and transforms them into a list of Document objects.

    Each document is formatted by combining the 'question', 'answer', 'title' and 'text' fields.

    Args:
        file_path (str): Path to the JSON file.

    Returns:
        List[Document]: List of documents ready to be indexed.
    '''
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)

    documents: List[Document] = []
    for item in data:
        combined_text = (
            f"Question: {item.get('question', '')}\n"
            f"Answer: {item.get('answer', '')}\n"
            f"Title: {item.get('title', '')}\n"
            f"Text: {item.get('text', '')}"
        )
        documents.append(Document(text=combined_text))
    return documents

Cargamos los documentos del archivo JSON:

In [13]:
documents: List[Document] = load_documents_from_json('hotpotqa_docs_reduced.json')

y hacemos un inspección básica:

In [ ]:
print('Cantidad de documentos: ', len(documents))

1000

Podemos ver qué forma tiene un documento:

In [15]:
documents[0]

Document(id_='c7d58736-23ff-41a4-8d70-5d23926ce6d5', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text="Question: Which magazine was started first Arthur's Magazine or First for Women?\nAnswer: Arthur's Magazine\nTitle: Radio City (Indian radio station)\nText: Radio City is India's first private FM radio station and was started on 3 July 2001.  It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).  It plays Hindi, English and regional songs.  It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007.  Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news

**Observaciones**:
- en el campo `text` podemos observar el `combined_text` que agregamos en la función, que contiene los cuatro campos mencionados.
- la clase `Document` tiene un parámetro `relationships`, que es un diccionario destinado a almacenar relaciones con otros documentos o nodos. En este contexto donde es necesario un razonamiento multi-hop parece bastante llamativo intentar modelar las relaciones entre los documentos. Sin embargo, los resultados son bastante bueno incluso sin utilizar este parámetro, como veremos más adelante.

---
### Indexing, Embedding y Almacenamiento

...un **index** es...

In [22]:
# Configurar los embeddings utilizando un modelo de HuggingFace
embed_model = HuggingFaceEmbedding(model_name='all-MiniLM-L6-v2')

Settings.embed_model = embed_model

In [ ]:
# Inicializar el cliente, seteando el path para almacenar los datos
db = chromadb.PersistentClient(path='./chroma_db')

# Crear la colección
chroma_collection = db.get_or_create_collection('hotpotqa_docs')

# Asignar chroma como el vector store por defecto en el contexto
vectore_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vectore_store)

# Crear el index
index = VectorStoreIndex.from_documents(
    documents=documents,
    storage_context=storage_context)

---
### Querying

Una de las piezas faltantes del sistema hasta ahora es el LLM. Podemos construirlo a través de la clase `OpenAI` configurando ciertos parámetros, y luego setearlo como el llm por defecto en los ajustes globales:

In [26]:
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

Settings.llm = OpenAI(
    api_key=OPENAI_API_KEY,
    model='gpt-4o',
    temperature=0)

En el código anterior, se ajustó el parámetro `temperature` a 0. Haciendo esto, le estamos indicando al LLM que sea lo más determinista posible. Tiene sentido en un contexto donde se usa RAG por varias razones:
- Un valor bajo de temperatura ayuda a que las respuestas se basen más fuertemente en la evidencia recuperada (los documentos indexados) en lugar de inventar o imaginar contenido adicional.
- Cuando se necesitan respuestas reproducibles y consistentes (por ejemplo, para pruebas o evaluaciones), la temperatura baja facilita obtener resultados más estables.
- El propóstio de RAG es complementar la capacidad de razonamiento del LLM con información externa confiable. Al hacer el modele menos creativo, reducimos el riesgo de que la generación se desvíe del material de referencia.

Básicamente buscamos que el LLM se base lo máximo posible en los documentos y no en lo que "él sabe".

Por último, creamos el motor de consultas, lo que va a permitir la interacción con el usuario para que este pueda hacer preguntas y recibir las respuestas.

En el contexto de LlamaIndex, un **query_engine** es una interfaz genérica que nos permite hacer preguntas sobre los datos. Este motor toma una consulta en lenguaje natural, y luego del proceso descrito anteriormente, devuelve una respuesta.

Para crear uno, convertimos el index en un motor de consultas de la siguiente manera:

In [25]:
# Convertir el índice en un query engine
query_engine = index.as_query_engine()

- **Observación**: Tanto el *retriever* como el *response synthesizer* van a estar implícitos. Esto no es estrictamente necesario, podemos construirlos y personalizarlos. Inicialemnte los dejamos así para observar los resultados y, en caso de ser necesario, podemos construirlos explícitamente para ajustar sus parámetros.

Cuando hacemos `index.as_query_engine()` sin especificar un retriever, LlamaIndex usa un retriever por defecto que depende del tipo de index que hayamos creado. En este caso, como usamos un `VectorStoreIndex`, va a usar un `VectoreStoreRetriever`.

Definimos la siguiente función que toma la consulta y genera la respuesta. Nos va a servir más adelante para la interfaz con `Gradio`. 

In [ ]:
def answer_question(query: str) -> str:
    '''
    Answers a question using the RAG pipeline.

    Args:
        query (str): The question to be answered.

    Returns:
        str: The generated answer.
    '''
    response = query_engine.query(query)
    return str(response)

Haciando consultas:

In [31]:
question = 'Which faith is designated to the University of Providence, private university accredited by the NW association of Schools and Colleges and located in a third largest city in Montana after being passed by Missoula?'
response = answer_question(question)
print('Question: ', question)
print('Response: ', response)

Question:  Which faith is designated to the University of Providence, private university accredited by the NW association of Schools and Colleges and located in a third largest city in Montana after being passed by Missoula?
Response:  Roman Catholic


Luego de probar varias preguntas del archivo `hotpotqa_docs_reduced_qa`, podemos comprobar que los resultados son bastante acertadas, en la mayoría de casos coinciden exactamente.

---
### Evaluación

---
### Referencias

- Documentación oficial de *LlamaIndex*: https://docs.llamaindex.ai/en/stable/
- ¿Qué es RAG?: https://aws.amazon.com/what-is/retrieval-augmented-generation/

--

- HotpotQA: https://hotpotqa.github.io/

--

- `json` — JSON encoder and decoder: https://docs.python.org/3/library/json.html
- `sentence-transformers/all-MiniLM-L6-v2`: https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2